# 03 — Deep Learning Models vs. Reference Baselines

### Academic Research Objectives:
1. **Comparative Benchmarking:** Evaluate multi-horizon contagion prediction across the complete model zoo:
   - **VAR-GFEVD:** Statistical econometric baseline.
   - **Independent Cascade:** Probabilistic epidemic diffusion baseline.
   - **CryptoLSTM:** Non-graph temporal recurrent baseline.
   - **CryptoTGN:** Continuous-time dynamic graph neural network with evolving memory.
   - **DyGFormer Feasibility:** Dynamic graph transformer comparison.
2. **Metrics:** Prioritize **AUPRC (Average Precision)** for node classification given severe class imbalance (<5% positive shocks), alongside AUROC, RMSE, and MAE for multi-horizon returns.
3. **Repeated Seeds & Statistical Testing:** Evaluate model stability across repeated random seeds.


In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

src_path = str(Path("../src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import torch
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from cryptograph.data.universe import TARGET_UNIVERSE_30
from cryptograph.models.tgn import CryptoTGN
from cryptograph.models.lstm import CryptoLSTM
from cryptograph.training.seed import seed_everything
from cryptograph.training.device import resolve_device

device = resolve_device("auto")
seed_everything(42)
print(f"Active Device: {device}")


## 1. Load Processed 30-Coin Hourly Dataset & Chronological Splits


In [ ]:
dataset_path = Path("../data/processed/v2/1h/processed_dataset.pt")
data = torch.load(dataset_path, map_location="cpu", weights_only=False)

features = data["features"]
targets_shock = data["targets_shock"]
targets_return = data["targets_return"]
validity = data["target_validity_mask"]
active_mask = data["active_mask"]
split_indices = data["split_indices"]

train_idx = np.array(split_indices["train_indices"])
val_idx = np.array(split_indices["val_indices"])
test_idx = np.array(split_indices["test_indices"])

print(f"Features: {features.shape}, Targets: {targets_shock.shape}")
print(f"Partitions: Train={len(train_idx)}, Val={len(val_idx)}, Test={len(test_idx)}")


## 2. Train and Evaluate Non-Graph Baseline (`CryptoLSTM`)


In [ ]:
# Model skeleton: CryptoLSTM
lstm_model = CryptoLSTM(
    input_dim=features.shape[-1],
    hidden_dim=64,
    num_layers=2,
    out_horizons=len(data["horizons"]),
    dropout=0.2
).to(device)

print(lstm_model)


## 3. Train and Evaluate Continuous-Time Dynamic Graph Network (`CryptoTGN`)


In [ ]:
# Model skeleton: CryptoTGN
tgn_model = CryptoTGN(
    num_nodes=len(data["asset_names"]),
    raw_msg_dim=features.shape[-1],
    memory_dim=64,
    time_dim=16,
    embedding_dim=64,
    out_horizons=len(data["horizons"]),
    dropout=0.2
).to(device)

print(tgn_model)


## 4. Multi-Horizon Performance Comparison & Empirical Leaderboard


In [ ]:
# Comparative Leaderboard Skeleton
print("Comparative Leaderboard (AUPRC / AUROC Across Horizons)")
